# Cherry Bootstrap Workflow

Run the bootstrapped labeling workflow cell by cell instead of as a script.

This notebook follows the same pipeline as the example runner:
1. Configure paths and thresholds
2. Run Grounding DINO
3. Prepare the YOLO dataset
4. Train YOLO
5. Run YOLO inference
6. Merge labels automatically or launch manual review

In [1]:
import importlib.metadata as metadata
import subprocess
import sys

transformers_version = metadata.version('transformers')
if int(transformers_version.split('.', 1)[0]) >= 5:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'transformers<5'])
    for module_name in list(sys.modules):
        if (
            module_name == 'transformers'
            or module_name.startswith('transformers.')
            or module_name.startswith('grounding_dino')
            or module_name.startswith('Labeling.grounding_dino')
        ):
            sys.modules.pop(module_name, None)
    print('Pinned transformers<5 for Grounding DINO compatibility.')
else:
    print(f'Using transformers {transformers_version}.')

Using transformers 4.57.6.


In [2]:
from pathlib import Path
import sys


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'Labeling').exists():
            return candidate
    raise FileNotFoundError('Could not find the repository root containing Labeling/')


REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from Labeling.config import BootstrapWorkflowConfig
from Labeling.manual_review import ManualReviewConfig, ManualReviewSession
from Labeling.grounding_dino import run_grounding_dino_directory
from Labeling.yolo import prepare_yolo_dataset, train_yolo, run_yolo_inference
from Labeling.merge import merge_label_sets

print(f'Repo root: {REPO_ROOT}')

Repo root: /Users/peter/Documents/MDS_UBC/My_Projects/zero-shot-labeling


## 1. Configure the workflow

Edit the values below before running the next cells.

The main things to adjust are:
- `image_dir`: where the input images live
- `text_prompt`: the phrase Grounding DINO should look for
- `box_threshold` and `text_threshold`: how strict the detector should be
- `train_*`: YOLO training settings
- `merge_mode`: choose automatic merging or manual review

In [ ]:
# Paths for inputs and outputs.
config = BootstrapWorkflowConfig(
    image_dir=REPO_ROOT / 'dataset' / 'cherry-detection-1' / 'valid' / 'images',
    dino_output_dir=REPO_ROOT / 'dataset' / 'dino_out',
    yolo_dataset_dir=REPO_ROOT / 'dataset' / 'yolo_data_set',
    yolo_project_dir=REPO_ROOT / 'yolo_train_out',
    yolo_prediction_dir=REPO_ROOT / 'dataset' / 'yolo_predictions',
    merged_label_dir=REPO_ROOT / 'dataset' / 'autolabels' / 'merged_labels',

    # Grounding DINO prompt and thresholds.
    text_prompt='small red fruit',
    box_threshold=0.35,
    text_threshold=0.25,
    confidence_threshold=0.0,

    # YOLO training and prediction run names.
    train_run_name='yolo_pt_bootstrap',
    prediction_run_name='yolo_pt_bootstrap_pred',

    # Single-class cherry detection.
    class_names=('cherry',),

    # Dataset split and filtering settings.
    train_ratio=0.8,
    random_seed=42,
    nms_iou_threshold=0.5,
    merge_iou_threshold=0.5,
    merge_mode='auto',

    # YOLO training hyperparameters.
    train_epochs=50,
    train_batch_size=32,
    train_imgsz=640,
    train_workers=8,
    train_device=None,

    # YOLO inference settings.
    inference_conf=0.03,
    inference_iou=0.30,

    # Grounded-SAM-2 model files bundled with the repo.
    grounding_dino_config=REPO_ROOT / 'Grounded-SAM-2' / 'grounding_dino' / 'groundingdino' / 'config' / 'GroundingDINO_SwinT_OGC.py',
    grounding_dino_checkpoint=REPO_ROOT / 'Grounded-SAM-2' / 'gdino_checkpoints' / 'groundingdino_swint_ogc.pth',
    device=None,
)

manual_review_config = ManualReviewConfig(
    image_source=REPO_ROOT / 'dataset' / 'cherry-detection-1' / 'valid' / 'images',
    dino_labels=REPO_ROOT / 'dataset' / 'dino_out' / 'labels',
    yolo_labels=REPO_ROOT / 'dataset' / 'yolo_predictions' / 'yolo_pt_bootstrap_pred' / 'labels',
    output_labels=REPO_ROOT / 'dataset' / 'autolabels' / 'manual_review_labels',
)

config

## 2. Run Grounding DINO

This writes labels and visualizations into `dataset/dino_out`.

In [4]:
dino_output_dir = run_grounding_dino_directory(
    image_dir=config.image_dir,
    output_dir=config.dino_output_dir,
    text_prompt=config.text_prompt,
    box_threshold=config.box_threshold,
    text_threshold=config.text_threshold,
    confidence_threshold=config.confidence_threshold,
    nms_iou_threshold=config.nms_iou_threshold,
    device=config.device,
    grounding_dino_config=config.grounding_dino_config,
    grounding_dino_checkpoint=config.grounding_dino_checkpoint,
)

dino_output_dir

final text_encoder_type: bert-base-uncased


PosixPath('/Users/peter/Documents/MDS_UBC/My_Projects/zero-shot-labeling/dataset/dino_out')

## 3. Prepare the YOLO dataset

This creates the train/valid split and `data.yaml`. 

Label_dir is the directory where the Grounding DINO labels are stored. It should be `dataset/dino_out/labels` if you ran the previous cell with the default paths.

In [5]:
data_yaml = prepare_yolo_dataset(
    image_dir=config.image_dir,
    label_dir=dino_output_dir / 'labels',
    yolo_data_dir=config.yolo_dataset_dir,
    class_names=config.class_names,
    train_ratio=config.train_ratio,
    seed=config.random_seed,
)

data_yaml

PosixPath('/Users/peter/Documents/MDS_UBC/My_Projects/zero-shot-labeling/dataset/yolo_data_set/data.yaml')

## 4. Train YOLO

This trains the model on the auto-generated labels.

In [ ]:
best_weights = train_yolo(
    data_yaml=data_yaml,
    project_dir=config.yolo_project_dir,
    run_name=config.train_run_name,
    epochs=config.train_epochs,
    imgsz=config.train_imgsz,
    batch_size=config.train_batch_size,
    resume=True,
    workers=config.train_workers,
    device=config.train_device or config.device,
)

best_weights

## 5. Run YOLO inference

This generates prediction labels that can be merged back with the DINO labels.

In [ ]:
prediction_dir = run_yolo_inference(
    weights_path=best_weights,
    source_dir=config.image_dir,
    project_dir=config.yolo_prediction_dir,
    run_name=config.prediction_run_name,
    conf=config.inference_conf,
    iou=config.inference_iou,
    device=config.device,
)

prediction_dir

## 6. Merge labels

Set `merge_mode` in the config to control the behavior:
- `auto`: automatically merge YOLO and DINO labels
- `manual`: launch the interactive review UI

In [ ]:
if config.merge_mode == 'auto':
    merged_dir = merge_label_sets(
        image_dir=config.image_dir,
        yolo_label_dir=prediction_dir / 'labels',
        dino_label_dir=dino_output_dir / 'labels',
        merged_label_dir=config.merged_label_dir,
        iou_threshold=config.merge_iou_threshold,
    )
    print(f'Automatic merge complete: {merged_dir}')
elif config.merge_mode == 'manual':
    ManualReviewSession(manual_review_config).run()
else:
    raise ValueError(f'Unsupported merge_mode: {config.merge_mode}')

## 7. Summary

You can inspect the generated outputs in the folders referenced by the config object.